In [3]:
from datasets import load_dataset

c:\Users\Nizwa\miniconda3\envs\ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ts_corpus = load_dataset("roneneldan/TinyStories")

c:\Users\Nizwa\miniconda3\envs\ai\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Nizwa\.cache\huggingface\hub\datasets--roneneldan--TinyStories. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating validation split: 100%|██████████| 21990/21990 [00:00<00:00, 1350861.12 examples/s]


In [4]:
ts_corpus

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})

In [5]:
ts_corpus["train"][0]["text"]

'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'

In [3]:
ts_instruct = load_dataset("roneneldan/TinyStoriesInstruct")

c:\Users\Nizwa\miniconda3\envs\ai\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Nizwa\.cache\huggingface\hub\datasets--roneneldan--TinyStoriesInstruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating validation split: 100%|██████████| 218380/218380 [00:00<00:00, 2005408.11 examples/s]


In [9]:
import re

all_texts = []
for split in ["train", "validation", "test"]:
    for item in ds[split]:
        all_texts.append(item["prompt"])
        all_texts.append(item["utterance"])

# find all patterns like _something_
pattern = re.compile(r"_\w+_")
placeholders = set()

for text in all_texts:
    matches = pattern.findall(text)
    placeholders.update(matches)

print("Found placeholders in dataset:")
print(sorted(placeholders))

Found placeholders in dataset:
['_1_comma_', '_COLON_', '_comma_', '_comma_000_comma_', '_comma_5_comma_', '_comma__comma_', '_comma__comma__comma_', '_comma__comma__comma__comma_', '_comma_cantelope_comma_', '_comma_finally_comma_', '_comma_green_comma_', '_comma_really_comma_']


In [15]:
ds["train"][1]

{'conv_id': 'hit:0_conv:1',
 'utterance_idx': 2,
 'context': 'sentimental',
 'prompt': 'I remember going to the fireworks with my best friend. There was a lot of people_comma_ but it only felt like us in the world.',
 'speaker_idx': 0,
 'utterance': 'Was this a friend you were in love with_comma_ or just a best friend?',
 'selfeval': '5|5|5_2|2|5',
 'tags': ''}

In [6]:
from easydict import EasyDict

test1 = EasyDict()
test1.name1 = "Test Config 1"

test2 = EasyDict()
test2.name2 = "Test Config 2"

test1.update(test2)

In [7]:
test1

{'name1': 'Test Config 1', 'name2': 'Test Config 2'}

In [ ]:
from model import generate_causal_mask
import torch

# 2 batches, 5 tokens each
input_ids = torch.tensor([[1, 0, 0, 2, 3, 4, 5, 0], [6, 7, 8, 9, 10, 0, 0, 0]])

input_ids.shape  # (B, L)

torch.Size([2, 8])

In [22]:
print(input_ids)

tensor([[ 1,  0,  0,  2,  3,  4,  5,  0],
        [ 6,  7,  8,  9, 10,  0,  0,  0]])


In [ ]:
mask = generate_causal_mask(input_ids.shape[1], input_ids.device)

mask.shape  # (1,1,L,L)

torch.Size([1, 1, 8, 8])

In [24]:
num_heads = 4

B, L = input_ids.shape
H = num_heads  # number of attention heads in your MHA

In [25]:
mask = mask.expand(B, H, L, L)

mask.shape

torch.Size([2, 4, 8, 8])

In [26]:
pad_id = 0

padding_mask = (input_ids != pad_id).unsqueeze(1).unsqueeze(2)  # (B, 1, 1, L)

In [27]:
# True = valid token, False = pad
mask = mask & padding_mask  # (B, H, L, L)

mask.shape

torch.Size([2, 4, 8, 8])

In [ ]:
mask

tensor([[[[ True, False, False, False, False, False, False, False],
          [ True, False, False, False, False, False, False, False],
          [ True, False, False, False, False, False, False, False],
          [ True, False, False,  True, False, False, False, False],
          [ True, False, False,  True,  True, False, False, False],
          [ True, False, False,  True,  True,  True, False, False],
          [ True, False, False,  True,  True,  True,  True, False],
          [ True, False, False,  True,  True,  True,  True, False]],

         [[ True, False, False, False, False, False, False, False],
          [ True, False, False, False, False, False, False, False],
          [ True, False, False, False, False, False, False, False],
          [ True, False, False,  True, False, False, False, False],
          [ True, False, False,  True,  True, False, False, False],
          [ True, False, False,  True,  True,  True, False, False],
          [ True, False, False,  True,  True, 

In [ ]:
B, L, V = 1, 4, 5  # batch size, sequence length, vocabulary size

logits = torch.randn(B, L, V)  # (B, L, V)
labels = torch.tensor([[1, 2, 3, 4]])  # (B, L)

In [ ]:
shift_logits = logits[:, :-1, :].contiguous()  # (B, L-1, V)
shift_labels = labels[:, 1:].contiguous()  # (B, L-1)

In [45]:
print(logits)

tensor([[[-0.2552,  1.7677, -1.3673, -0.5024, -1.5271],
         [-0.1959,  0.0762, -0.8660, -0.3407,  0.7085],
         [-0.9155,  1.5716, -0.0453,  1.0679, -0.4274],
         [ 1.5407,  1.6160, -1.4435, -0.1791, -0.5408]]])


In [44]:
print(shift_logits)

tensor([[[-0.2552,  1.7677, -1.3673, -0.5024, -1.5271],
         [-0.1959,  0.0762, -0.8660, -0.3407,  0.7085],
         [-0.9155,  1.5716, -0.0453,  1.0679, -0.4274]]])


In [46]:
print(shift_labels)

tensor([[2, 3, 4]])


In [ ]:
# Number of stories to test with
num_test_stories = 5

story_data = []
current_story = {"features": "", "words": "", "summary": "", "story": []}
stories_collected = 0

for line in ts_instruct["train"]["text"]:
    line = line.strip()

    if line.startswith("Features:"):
        current_story["features"] = line[len("Features: ") :]
    elif line.startswith("Words:"):
        current_story["words"] = line[len("Words: ") :]
    elif line.startswith("Summary:"):
        current_story["summary"] = line[len("Summary: ") :]
    elif line == "Story:":
        current_story["story"] = []
    elif line == "<|endoftext|>":
        current_story["story"] = " ".join(current_story["story"]).strip()
        story_data.append(current_story)
        stories_collected += 1
        if stories_collected >= num_test_stories:
            break  # stop after collecting a few stories
        current_story = {"features": "", "words": "", "summary": "", "story": []}
    else:
        current_story["story"].append(line)

print(f"Collected {len(story_data)} stories for testing.")
print(story_data[0])

In [ ]:
def make_structured_data(data):
    cleaned_data = []
    current_story = {"features": "", "words": "", "summary": "", "story": []}

    for line in data:
        line = line.strip()

        # features: conditioning output to express story
        if line.startswith("Features:"):
            current_story["features"] = line[len("Features: ") :]
        # words: conditioning output to contain specified words
        elif line.startswith("Words:"):
            current_story["words"] = line[len("Words: ") :]
        # summary: the story prompt
        elif line.startswith("Summary:"):
            current_story["summary"] = line[len("Summary: ") :]
        # story: generated story based on prompt and conditioning
        elif line == "Story:":
            current_story["story"] = []
        # end of row
        elif line == "<|endoftext|>":
            current_story["story"] = " ".join(current_story["story"]).strip()
            cleaned_data.append(current_story)
            # reset for next data
            current_story = {"features": "", "words": "", "summary": "", "story": []}
        else:
            current_story["story"].append(line)

    return cleaned_data

In [4]:
ts_instruct = load_dataset("roneneldan/TinyStoriesInstruct")

In [ ]:
ts_instruct

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 21755681
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 218380
    })
})

In [9]:
ts_instruct["train"][:20]["text"]

['Features: Dialogue',
 'Words: quit, oak, gloomy',
 'Summary: Sara and Ben were playing in the park, but Sara wanted to go home because it was cold and dark. Ben convinced her to stay and play, but eventually agreed to go home and have hot cocoa.',
 'Story: ',
 '',
 'Sara and Ben were playing in the park. They liked to climb the big oak tree and pretend they were birds. They made nests with leaves and twigs and sang songs.',
 'But today, the sky was gloomy and the wind was cold. Sara felt sad and cold. She wanted to go home and have some hot cocoa.',
 '"Ben, I want to quit," she said. "It\'s too cold and dark. Let\'s go home."',
 'Ben looked at Sara and frowned. He liked the oak tree and the park. He wanted to stay and play.',
 '"No, Sara, don\'t quit," he said. "It\'s fun here. Look, there\'s a squirrel. Let\'s chase it."',
 "Sara shook her head. She didn't want to chase the squirrel. She wanted to go home and have some hot cocoa.",
 '"Please, Ben, let\'s go home," she said. "We can 

In [10]:
sft_data = make_structured_data(ts_instruct["train"]["text"])

In [13]:
sft_data[1]

{'features': 'Dialogue, BadEnding, MoralValue',
 'words': 'ride, work, upset',
 'summary': "Lily steals a new bike from a store and gets into an accident while riding it, resulting in her getting hurt and being sent to jail, losing her parents' trust and love. The moral of the story is not to steal.",
 'story': "Lily liked to ride her bike. She rode it every day after work. Work was a place where she helped her mom and dad with chores. She liked work, but she liked riding her bike more. One day, she saw a new bike in the store. It was shiny and red and had a bell. Lily wanted the new bike very much. She asked her mom and dad if they could buy it for her. They said no. They said the new bike was too expensive and that she already had a good bike. Lily was upset. She did not listen to her mom and dad. She thought they were mean and unfair. She decided to take the new bike without paying. She waited until the store was busy and then she sneaked out with the bike. She rode the new bike ver

In [12]:
len(sft_data)

2476532

In [10]:
import torch

token_A = [100, 200, 300, 350, 360]
token_B = [400, 500, 600]
constant = 20

token_A + [constant] + token_B

[100, 200, 300, 350, 360, 20, 400, 500, 600]

In [11]:
labels = (
    [-100] * len(token_A) + [constant] + token_B
)

labels

[-100, -100, -100, -100, -100, 20, 400, 500, 600]

In [14]:
2e-5 == 0.00002

True

In [1]:
import numpy as np
from datasets import load_dataset
from tokenizers import Tokenizer
from tqdm import tqdm

def analyze_token_lengths(tokenizer_path: str, corpus_name: str):
    tokenizer = Tokenizer.from_file(tokenizer_path)
    ds = load_dataset(corpus_name, split="train")

    print(f"total samples: {len(ds)}")

    lengths = []
    for item in tqdm(ds, desc="tokenizing"):
        text = item.get("text", "")
        if text:
            ids = tokenizer.encode(text).ids
            lengths.append(len(ids))

    lengths = np.array(lengths)

    print(f"\n--- token length analysis ---")
    print(f"samples analyzed : {len(lengths)}")
    print(f"min              : {lengths.min()}")
    print(f"max              : {lengths.max()}")
    print(f"mean             : {lengths.mean():.0f}")
    print(f"median           : {np.median(lengths):.0f}")
    print(f"std              : {lengths.std():.0f}")
    print(f"25th percentile  : {np.percentile(lengths, 25):.0f}")
    print(f"75th percentile  : {np.percentile(lengths, 75):.0f}")
    print(f"90th percentile  : {np.percentile(lengths, 90):.0f}")
    print(f"95th percentile  : {np.percentile(lengths, 95):.0f}")
    print(f"99th percentile  : {np.percentile(lengths, 99):.0f}")

    print(f"\n--- truncation coverage ---")
    for cutoff in [256, 512, 768, 850, 1024, 1280, 1536, 2048]:
        coverage = (lengths <= cutoff).mean() * 100
        print(f"max_seq={cutoff:<6} covers {coverage:.1f}% of samples")

c:\Users\Nizwa\miniconda3\envs\ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
analyze_token_lengths(
    tokenizer_path=".output/tokenizer.json",
    corpus_name="roneneldan/TinyStories",
)

total samples: 2119719


tokenizing: 100%|██████████| 2119719/2119719 [06:43<00:00, 5254.80it/s]



--- token length analysis ---
samples analyzed : 2119489
min              : 6
max              : 1314
mean             : 217
median           : 186
std              : 106
25th percentile  : 160
75th percentile  : 227
90th percentile  : 349
95th percentile  : 448
99th percentile  : 636

--- truncation coverage ---
max_seq=256    covers 82.1% of samples
max_seq=512    covers 97.1% of samples
max_seq=768    covers 99.7% of samples
max_seq=850    covers 99.8% of samples
max_seq=1024   covers 100.0% of samples
max_seq=1280   covers 100.0% of samples
max_seq=1536   covers 100.0% of samples
max_seq=2048   covers 100.0% of samples
